In [1]:
import pandas as pd

rfm = pd.read_csv("../data/rfm.csv")

rfm["R"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)   # recent = high score
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm[["Recency", "Frequency", "Monetary", "R", "F", "M"]].head(10)

,Recency,Frequency,Monetary,R,F,M
0,529,11,372.86,1,5,2
1,2,8,4921.53,5,4,5
2,75,5,1658.40,3,4,4
3,19,3,3678.69,5,3,5
4,310,1,294.40,2,1,2
5,375,1,300.93,2,1,2
6,36,9,1729.54,4,5,4
7,204,2,406.76,2,2,2
8,232,1,1079.40,2,1,3
9,214,2,947.61,2,2,3


In [2]:
def segment(r):
    if r.R >= 4 and r.F >= 4: return "Champions"
    if r.R >= 3 and r.F >= 3: return "Loyal"
    if r.R >= 4 and r.F <= 2: return "New / Promising"
    if r.R <= 2 and r.F >= 4: return "Cant Lose Them"
    if r.R <= 2 and r.F >= 3: return "At Risk"
    if r.R <= 2 and r.F <= 2: return "Hibernating / Lost"
    return "Need Attention"

rfm["Segment"] = rfm.apply(segment, axis=1)

In [3]:
summary = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum"),
).round(1).sort_values("Total_Revenue", ascending=False)

summary

,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary,Total_Revenue
Segment,,,,,
Champions,1477,20.4,15.6,8004.7,11822980.9
Loyal,1212,78.6,5.4,2000.8,2424925.2
Cant Lose Them,355,345.3,7.6,2894.7,1027620.1
Hibernating / Lost,1515,458.1,1.2,408.4,618778.3
At Risk,472,387.6,3.0,1025.9,484235.0
New / Promising,450,28.2,1.5,551.0,247942.3
Need Attention,380,106.6,1.4,528.8,200943.3


In [4]:
rfm.to_csv("../data/rfm_segments.csv", index=False)